# TripAdvisor Sentiment Analysis — Step 1: Web Scraping

Welcome to the first stage of your research project! In this notebook, we will extract raw hotel guest reviews directly from TripAdvisor.

### Learning Objectives:
1. Understand the concept of **Web Scraping**.
2. Learn how to extract HTML elements (titles, texts, ratings) using `rvest`.
3. Handle anti-bot protections and implement fallback datasets.

---

### ☁️ Google Colab Setup
If you are running this notebook in **Google Colab**, simply run the code cell below to download the dataset and install any missing tools. If you are running this locally in RStudio, you can skip it!

In [ ]:
# =====================================================================
# GOOGLE COLAB SETUP (Run this cell only if you are using Google Colab)
# =====================================================================
if (dir.exists("/content")) {
  if (!dir.exists("/content/MrKadekProject")) {
    cat("Downloading project files from GitHub...\n")
    system("git clone https://github.com/divanahadyan1618/MrKadekProject.git /content/MrKadekProject", ignore.stdout=TRUE, ignore.stderr=TRUE)
  }
  setwd("/content/MrKadekProject")
  
  # Install required packages if missing
  packages <- c("rvest", "tidytext", "stopwords", "syuzhet", "wordcloud", "RColorBrewer")
  new_packages <- packages[!(packages %in% installed.packages()[,"Package"])]
  if(length(new_packages)) install.packages(new_packages, quiet = TRUE)
  
  cat("Colab Environment Ready!\n")
}


## 1. Load Required Packages

We need several libraries to read web pages and manipulate the downloaded data.

In [ ]:
# =====================================================================
# TripAdvisor Sentiment Analysis - Step 1: Web Scraping
# =====================================================================
# Welcome! If you have never programmed before, do not worry.
# We will explain exactly what the computer is doing at every single step.
#
# PURPOSE OF THIS SCRIPT:
# We want to go to TripAdvisor, find a specific hotel, and automatically
# copy-paste hundreds of reviews into an Excel-like table so we can analyze them.
# This automatic copying is called "Web Scraping".

## 2. Define Target URL & Scraper Function

We define the TripAdvisor URL we want to scrape. We will build a custom function `scrape_tripadvisor_reviews()` that identifies specific **CSS classes** (like `span.yUiMA` or `span.ui_bubble_rating`) to pull exactly the text we want.

In [ ]:
# =====================================================================
# STEP 1: Load Required Packages (Adding New Tools to R)
# =====================================================================
# Think of R as a blank toolbox. "Packages" are specialized toolsets created
# by other programmers that we borrow to make our job easier.
# The 'library()' command tells the computer to open and equip that toolset.

# The 'rvest' toolset helps us download and read web pages.
library(rvest)
# The 'tidyverse' toolset helps us sort, filter, and organize data.
library(tidyverse)
# The 'xml2' toolset safely reads the raw computer code of a website.
library(xml2)

# 'cat()' is a command that simply prints a message on your screen.
cat("Scraping packages loaded successfully!\n")

## 3. Execute Scraper & Handle Fallback Data

Because TripAdvisor frequently blocks bots using Cloudflare, we execute our scraper inside a conditional block. If it fails to scrape live data, it will automatically initialize a realistic, 1000-row synthetic dataset so that our research can continue seamlessly.

In [ ]:
# =====================================================================
# STEP 2: Define Where We Want to Go
# =====================================================================
# The '<-' arrow is called the "assignment operator". 
# It means "take the internet link on the right, and save it inside a box named 'hotel_url' on the left."
hotel_url <- "https://www.tripadvisor.com/Hotel_Review-g297698-d607449-Reviews-Bvlgari_Resort_Bali-Uluwatu_Bukit_Peninsula_Bali.html"

## 4. Export Raw Data

Finally, we save the downloaded/generated data locally. We use `write_excel_csv2` to output a semicolon-delimited CSV, ensuring it opens perfectly in Indonesian and European versions of Microsoft Excel.

In [ ]:
# =====================================================================
# STEP 3: Create the "Scraper" Machine
# =====================================================================
# Here we are building our own custom tool (a "function") named 'scrape_tripadvisor_reviews'.
# Think of it like a recipe: we give it a web link (url), and it gives us a table of reviews back.
scrape_tripadvisor_reviews <- function(url) {
  
  # 'tryCatch' tells the computer: "Try to do this, but if the website blocks you, don't crash!"
  tryCatch({
    # 1. Download the webpage. 'user_agent' disguises our computer so TripAdvisor thinks we are a normal Chrome browser.
    page <- read_html(x = url, options = "RECOVER", 
                      user_agent = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36")
    
    # The '%>%' symbol means "AND THEN". 
    # Example: Take the page AND THEN find the text AND THEN clean it up.
    
    # 2. Extract the Review Text by looking for TripAdvisor's specific font/design codes ("span.yUiMA")
    reviews <- page %>% html_nodes("span.yUiMA, div.fIrGe") %>% html_text(trim = TRUE)
    
    # 3. Extract the Review Titles (the bold headline of the review)
    titles <- page %>% html_nodes("span.yUiMA, a.QYdug") %>% html_text(trim = TRUE)
    
    # 4. Extract the Star Ratings ( TripAdvisor stores 5 stars as "50", so we divide by 10 )
    ratings <- page %>% html_nodes("span.ui_bubble_rating") %>% html_attr("class") %>% 
      str_extract("[0-9]+") %>% as.numeric() / 10
    
    # 5. Extract the Date the guest stayed at the hotel
    dates <- page %>% html_nodes("span.teSgY") %>% html_text(trim = TRUE) %>% 
      str_remove("Date of stay: ")
    
    # 6. 'tibble' means "Table". We are taking the 4 lists above and taping them together into a spreadsheet.
    df <- tibble(
      title = head(titles, length(reviews)),
      review_text = reviews,
      rating = head(ratings, length(reviews)),
      review_date = head(dates, length(reviews))
    )
    
    # Return the finished table back to us
    return(df)
    
  }, error = function(e) {
    # If the website blocks us, print a warning message instead of crashing.
    message("Warning: Live scraping was blocked or interrupted: ", e$message)
    return(NULL)
  })
}

## 🎓 Student Exercise / Assignment Connection

For your project assignment, make sure that:
1. You identify the correct `hotel_url` for the exact hotel you want to analyze.
2. Be aware that the CSS classes `span.yUiMA` might change over time as TripAdvisor updates its website design.

**Great job! Open `02_cleaning.ipynb` to clean the raw data.**